Start from the dataset kickstarter-14-04, take out the unnecessary columsn that I don't know why are there:

In [68]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import nltk
from nltk.corpus import stopwords
import string 
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
import re 
from collections import Counter
import ast
import glob

In [69]:
campaigns = pd.read_csv('raw_kickstarter.csv', index_col=0)
campaigns

,url,title,description,pledged,usd_pledged,converted_pledged_amount,goal,currency
0,https://www.kickstarter.com/projects/thetruthb...,NaN,The PROBLEM: So much entertainment today pushe...,71123.0,71123.0,61607,48000.0,USD
1,https://www.kickstarter.com/projects/99625582/...,NaN,Millions of American college students have stu...,65318.0,65318.0,56579,61500.0,USD
2,https://www.kickstarter.com/projects/distortre...,Cartoon Network Alphabet Pins,Full A-Z Set I'm launching this set to show my...,462.0,462.0,400,8000.0,USD
3,https://www.kickstarter.com/projects/jordym/th...,The Balloon - a short film,"On a sleepy summer afternoon, we stared into t...",5137.0,5137.0,4449,15000.0,USD
4,https://www.kickstarter.com/projects/trans-mov...,NaN,48 hours of pledge matching! Amazing news! Two...,50640.0,50640.0,43865,50000.0,USD
...,...,...,...,...,...,...,...,...
8390,https://www.kickstarter.com/projects/zerwcolla...,ZerwCollab: Your AI-Powered App Builder in Sec...,🚀 What If You Could Launch a Website or App… U...,2128.0,2128.0,1823,35900.0,USD
8391,https://www.kickstarter.com/projects/jacobzirk...,VFX Oasis: Free VFX Assets and Footage,What is VFX Oasis? VFX Oasis is an industry fo...,580.0,580.0,496,6000.0,USD
8392,https://www.kickstarter.com/projects/153189407...,Sharp X - Edge Gadget,Kickstarter Project Story – USA-Market Sharpen...,857.0,857.0,734,7800.0,USD
8393,https://www.kickstarter.com/projects/diyengine...,Steady Shot Bot: Hyperlapse + Steadycam + Auto...,"Why As a hobbyist, I recently found myself wan...",6645.0,6645.0,5693,310000.0,USD


### as a very first step we add the categories to each project

In [70]:

folder_path = "Kickstarter_2026-03-12T03_20_26_556Z"

files = glob.glob(f"{folder_path}/*.csv")

print("Number of files found:", len(files))

dfs = []

for file in files:
    print("Loading:", file)
    df_temp = pd.read_csv(file)
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)



url_to_category = {}
for _, row in df.iterrows():
    parsed = ast.literal_eval(row['category'])
    parent_name = parsed.get('parent_name') or parsed.get('name')
    url_parsed = ast.literal_eval(row['urls'])
    project_url = url_parsed['web']['project']
    url_to_category[project_url] = parent_name


campaigns['category'] = campaigns['url'].map(url_to_category)

print(f"Matched: {campaigns['category'].notna().sum()} / {len(campaigns)}")
print(campaigns['category'].value_counts())

Number of files found: 85
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter001.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter002.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter003.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter004.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter005.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter006.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter007.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter008.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter009.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter010.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter011.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter012.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter013.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter014.csv
Lo

take out the unnecessary columns, add the reached one and classify as success or failure (maybe it's already in the initial dataset but honestly I don't remember)

In [71]:
campaigns['reached'] = (campaigns['pledged'] / campaigns['goal']) * 100

median_reached = campaigns['reached'].median()
campaigns['status'] = campaigns['reached'].apply(lambda x: 1 if x >= median_reached else 0)

Preprocessing: usual preprocessing stuff like lowercasing, taking out links, only keepin alpha numeric characters, tokenizing the words, and taking out the english stopwords, also took out the words that are less than 2 characters

In [72]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text_p = "".join([char for char in text if char.isalnum() or char.isspace()])
    
    words = word_tokenize(text_p) 
    
    
    stop_words = stopwords.words('english')
    words = [w for w in words if w not in stop_words and len(w) > 2 and w.isalpha()]
    filtered_words = [word for word in words if word not in stop_words] 
    
    return filtered_words  

campaigns['description_processed'] = campaigns['description'].apply(preprocess)

How do we decide which stopwords to include beside the 'normal' stopwords? We also want to include words that are very generic, not informative at all and coudl potentially appear in any kind of campaign, regardless of the topics in it (some examples could be the words 'kickstarter', campaign, etc etc)
Initially I thought TF-IDF would make sense but it doesn't actually, because TF-IDF both depends on how frequent a word is in each document and how frequent are documents that contain that words in the whole set of documents. (also you cannot average the TF-IDF of a word and pick the ones with lowest value because a rare word, which we wanna keep, could end up having a low TF-IDF value if it appears in few documents but many time in those few documents)
So potentially we can just use document frequency, meaning if the word appears in > alpha% of documents, we count it as a stopword?

Also quick sidenote: Other methods like BytePair encoding, WordPiece or SentencePiece are not really useful here because they look at subwords, and for example if we get 'kickstarter' and split it into kick and starter, then we might consider to keep kick because it makes sense with sport but we might not have any document where the word is present for real

This below just basically gives a counter of the document frequency of each token in our whole corpus of all the descriptions 

We apply lemmatization (I think probably better than stemming since it returns the lemma version of the word, instead of just cutting the ending of the word). We apply it now, before taking out the domain specific stopwords, because otherwise we could end up having to account for different versions of the same stopword (example: project vs projects)

In [73]:
lemmatizer = WordNetLemmatizer()

campaigns['lemmatized'] = campaigns['description_processed'].apply(
    lambda words: [lemmatizer.lemmatize(word) for word in words])


In [53]:
docs = campaigns['lemmatized']
N = len(docs)

df_counter = Counter()

for doc in docs:
    df_counter.update(set(doc))

df_table = pd.DataFrame({
    'word': list(df_counter.keys()),
    'doc_freq': list(df_counter.values())
})

df_table['doc_freq_ratio'] = df_table['doc_freq'] / N
df_table = df_table.sort_values('doc_freq_ratio', ascending=False)



In [76]:
df_table

,word,doc_freq,doc_freq_ratio
0,falling,54,0.026852
1,test,149,0.074092
2,platform,305,0.151666
3,production,1333,0.662854
4,college,322,0.160119
...,...,...,...
20005,144,1,0.000497
20006,snuggle,1,0.000497
20007,pediatrics,1,0.000497
20008,mocap,1,0.000497


Now the only method and the one that made most sense to me to take out too common words is basically just defining a threshold and see which words are too rare (like if it appears only 2-3 times in the whole dataset) or too much (in this case too much can be like appearing in more than 40-60% of the descriptions), you can try different values of the two initial thresholds (the one I saw looks the best would be 0.0005 for the low and 0.55 for the high)

In [75]:
min_doc_count = 3
max_doc_count = 5000  
min_ratio = min_doc_count / N
#max_ratio = max_doc_count / N
#min_ratio = 0.0002
max_ratio = 0.50
vocab = set(
    df_table[
        (df_table['doc_freq_ratio'] > min_ratio) &
        (df_table['doc_freq_ratio'] < max_ratio)
    ]['word']
)

print(df_table[df_table['doc_freq_ratio'] >= max_ratio])
df_table[df_table['doc_freq_ratio'] <= min_ratio]


           word  doc_freq  doc_freq_ratio
3    production      1333        0.662854
125        take      1045        0.519642
204        love      1062        0.528095
207         see      1014        0.504227
224    director      1024        0.509199
302        team      1024        0.509199
309        film      1689        0.839881
587        many      1012        0.503232


,word,doc_freq,doc_freq_ratio
46,tractor,3,0.001492
54,faithfulness,1,0.000497
70,modification,3,0.001492
73,119,1,0.000497
78,matchmaking,3,0.001492
...,...,...,...
20005,144,1,0.000497
20006,snuggle,1,0.000497
20007,pediatrics,1,0.000497
20008,mocap,1,0.000497


### Now we work on the data specific category by category 

In [77]:
# to actually take the overall common words out CAREFUL TO RUN 
campaigns['description_processed'] = campaigns['lemmatized'].apply(
    lambda doc: [w for w in doc if w in vocab]
)
campaigns = campaigns.drop(columns=['lemmatized'])

In [78]:
df = campaigns

#### 1. Identification of words appearing in more than 40% of documents

In [79]:
if "description_filtered" not in df.columns:
    df["description_filtered"] = None


df_film = df[df['category'] == 'Film & Video'].copy()
docs = df_film["description_processed"]

N = len(docs)
df_counter = Counter()

for doc in docs:
    df_counter.update(set(doc))

df_table = pd.DataFrame({
    "word": list(df_counter.keys()),
    "doc_freq": list(df_counter.values())
})

df_table["doc_freq_ratio"] = df_table["doc_freq"] / N
high_freq_words = df_table[df_table["doc_freq_ratio"] > 0.5]
high_freq_words = high_freq_words.sort_values("doc_freq_ratio", ascending = False)
high_freq_words

,word,doc_freq,doc_freq_ratio


In [80]:
extra_stopwords_film = set(high_freq_words["word"].tolist())

film_mask = df["category"] == "Film & Video"

df.loc[film_mask, "description_filtered"] = df.loc[film_mask, "description_processed"].apply(
    lambda text: [word for word in text if word not in extra_stopwords_film]
)

df

,url,title,description,pledged,usd_pledged,converted_pledged_amount,goal,currency,category,reached,status,description_processed,description_filtered
0,https://www.kickstarter.com/projects/thetruthb...,NaN,The PROBLEM: So much entertainment today pushe...,71123.0,71123.0,61607,48000.0,USD,Film & Video,148.172917,1,"[problem, much, entertainment, today, push, un...","[problem, much, entertainment, today, push, un..."
1,https://www.kickstarter.com/projects/99625582/...,NaN,Millions of American college students have stu...,65318.0,65318.0,56579,61500.0,USD,Film & Video,106.208130,1,"[million, american, college, student, studied,...","[million, american, college, student, studied,..."
2,https://www.kickstarter.com/projects/distortre...,Cartoon Network Alphabet Pins,Full A-Z Set I'm launching this set to show my...,462.0,462.0,400,8000.0,USD,Film & Video,5.775000,0,"[full, set, launching, set, show, early, carto...","[full, set, launching, set, show, early, carto..."
3,https://www.kickstarter.com/projects/jordym/th...,The Balloon - a short film,"On a sleepy summer afternoon, we stared into t...",5137.0,5137.0,4449,15000.0,USD,Film & Video,34.246667,0,"[sleepy, summer, afternoon, stared, void, floa...","[sleepy, summer, afternoon, stared, void, floa..."
4,https://www.kickstarter.com/projects/trans-mov...,NaN,48 hours of pledge matching! Amazing news! Two...,50640.0,50640.0,43865,50000.0,USD,Film & Video,101.280000,0,"[hour, pledge, matching, amazing, news, two, g...","[hour, pledge, matching, amazing, news, two, g..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8390,https://www.kickstarter.com/projects/zerwcolla...,ZerwCollab: Your AI-Powered App Builder in Sec...,🚀 What If You Could Launch a Website or App… U...,2128.0,2128.0,1823,35900.0,USD,Technology,5.927577,0,"[could, launch, website, app, using, voice, im...",None
8391,https://www.kickstarter.com/projects/jacobzirk...,VFX Oasis: Free VFX Assets and Footage,What is VFX Oasis? VFX Oasis is an industry fo...,580.0,580.0,496,6000.0,USD,Technology,9.666667,0,"[vfx, oasis, vfx, oasis, industry, focused, vi...",None
8392,https://www.kickstarter.com/projects/153189407...,Sharp X - Edge Gadget,Kickstarter Project Story – USA-Market Sharpen...,857.0,857.0,734,7800.0,USD,Technology,10.987179,0,"[real, today, modern, come, especially, contin...",None
8393,https://www.kickstarter.com/projects/diyengine...,Steady Shot Bot: Hyperlapse + Steadycam + Auto...,"Why As a hobbyist, I recently found myself wan...",6645.0,6645.0,5693,310000.0,USD,Technology,2.143548,0,"[recently, found, wanting, photography, techno...",None


In [81]:
any("film" in text for text in df.loc[df["category"] == "Film & Video", "description_filtered"]) #use to check if it actually took out stopwords like 'film'

False

In [82]:
df['category'].unique()

array(['Film & Video', 'Games', 'Music', 'Publishing', 'Technology'],
      dtype=object)

In [83]:

selected_categories = ['Technology', 'Games', 'Music', 'Publishing', 'Film & Video']
category_thresholds = [0.45, 0.53, 0.40, 0.40, 0.40]  
category_high_freq_words = {}

for category_name, category_threshold in zip(selected_categories, category_thresholds):
    category_df = df[df["category"] == category_name].copy()
    docs = category_df["description_processed"]
    if docs.empty:
        print(f"{category_name}: no documents found")
        continue

    category_counter = Counter()
    for doc in docs:
        category_counter.update(set(doc))

    category_table = pd.DataFrame({
        "word": list(category_counter.keys()),
        "doc_freq": list(category_counter.values())
    })

    category_table["doc_freq_ratio"] = category_table["doc_freq"] / len(docs)
    high_freq_words = category_table[category_table["doc_freq_ratio"] > category_threshold]
    high_freq_words = high_freq_words.sort_values("doc_freq_ratio", ascending=False)

    category_high_freq_words[category_name] = high_freq_words
    print(f"\n{category_name}")
    display(high_freq_words)

category_high_freq_words


Technology


,word,doc_freq,doc_freq_ratio
316,use,999,0.670920
282,design,905,0.607790
116,product,854,0.573539
66,experience,813,0.546004
152,designed,802,0.538617
45,feature,779,0.523170
142,even,756,0.507723
9,using,746,0.501007
286,without,745,0.500336
296,every,727,0.488247



Games


,word,doc_freq,doc_freq_ratio
36,game,1199,0.843179
130,play,1018,0.715893
232,every,875,0.615331
1121,player,864,0.607595
553,campaign,827,0.581575
73,experience,813,0.571730
206,back,807,0.567511
523,design,800,0.562588
9,come,772,0.542897
266,reward,768,0.540084



Music


...


Publishing


,word,doc_freq,doc_freq_ratio
70,book,938,0.772652
174,every,613,0.504942
4,come,587,0.483526
927,reward,583,0.480231
814,page,583,0.480231
95,art,568,0.467875
149,cover,557,0.458814
433,two,552,0.454695
270,even,551,0.453871
193,campaign,546,0.449753



Film & Video


,word,doc_freq,doc_freq_ratio
336,short,995,0.494779
340,show,983,0.488812
48,making,969,0.481850
239,crew,959,0.476877
746,producer,934,0.464446
937,come,933,0.463948
1053,bring,926,0.460467
157,experience,925,0.459970
244,day,906,0.450522
81,every,903,0.449030


{'Technology':            word  doc_freq  doc_freq_ratio
 316         use       999        0.670920
 282      design       905        0.607790
 116     product       854        0.573539
 66   experience       813        0.546004
 152    designed       802        0.538617
 45      feature       779        0.523170
 142        even       756        0.507723
 9         using       746        0.501007
 286     without       745        0.500336
 296       every       727        0.488247
 216      create       713        0.478845
 114        come       708        0.475487
 51   technology       701        0.470786
 263      system       672        0.451310,
 'Games':             word  doc_freq  doc_freq_ratio
 36          game      1199        0.843179
 130         play      1018        0.715893
 232        every       875        0.615331
 1121      player       864        0.607595
 553     campaign       827        0.581575
 73    experience       813        0.571730
 206         back      

In [84]:
# Domain-specific stopwords removal, run only after reviewing the preview above
category_stopwords = {}

for category_name in selected_categories:
    high_freq_words = category_high_freq_words.get(category_name)

    extra_stopwords = set(high_freq_words["word"].tolist())
    category_stopwords[category_name] = extra_stopwords

    category_mask = df["category"] == category_name
    df.loc[category_mask, "description_filtered"] = df.loc[category_mask, "description_processed"].apply(
        lambda text, stopwords=extra_stopwords: [word for word in text if word not in stopwords]
    )

category_stopwords

{'Technology': {'come',
  'create',
  'design',
  'designed',
  'even',
  'every',
  'experience',
  'feature',
  'product',
  'system',
  'technology',
  'use',
  'using',
  'without'},
 'Games': {'back',
  'campaign',
  'come',
  'create',
  'design',
  'even',
  'every',
  'experience',
  'game',
  'play',
  'player',
  'reward',
  'set'},
 'Music': {'album',
  'artist',
  'come',
  'cost',
  'friend',
  'know',
  'much',
  'music',
  'musician',
  'part',
  'record',
  'recording',
  'release',
  'share',
  'song',
  'sound',
  'studio',
  'thank',
  'together',
  'video'},
 'Publishing': {'art',
  'author',
  'back',
  'book',
  'bring',
  'campaign',
  'come',
  'cost',
  'cover',
  'even',
  'every',
  'experience',
  'find',
  'know',
  'much',
  'page',
  'part',
  'print',
  'reward',
  'two',
  'would'},
 'Film & Video': {'art',
  'best',
  'bring',
  'cast',
  'character',
  'come',
  'create',
  'crew',
  'day',
  'even',
  'every',
  'experience',
  'feature',
  'festival

In [86]:
# only at the end 
df.to_csv('Kickstarter_processed.csv')

In [85]:
# Total number of unique tokens in the final corpus
print(df['description_filtered'].isna().sum())
final_docs = df["description_filtered"].dropna()


unique_tokens = set(token for doc in final_docs for token in doc)
print("Total unique tokens:", len(unique_tokens))

0
Total unique tokens: 13675
